# Hi-cGAN Diffusion Model — Google Colab Training & Prediction Notebook
This notebook clones the repository (including the 250kb RNA matrices in `hicgan/data/rna_gt/chr*.npz` and `splits.txt`) directly from GitHub (`diffusion` branch), trains the Diffusion model where **1 fixed random Gaussian matrix per chromosome** (`hicgan/data/random_input/chr*.npy`) is used as input, and evaluates predictions with `predict.py`.

## 1. Check GPU & Clone Repository from GitHub

In [ ]:
!nvidia-smi

import os

REPO_URL = 'https://github.com/nehirvardar/Hi-cGAN.git'
BRANCH = 'diffusion'
PROJECT_DIR = '/content/Hi-cGAN'

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull origin {BRANCH}

os.chdir(PROJECT_DIR)
print('Current working directory:', os.getcwd())

## 2. Install Dependencies & Verify GitHub Data (`hicgan/data/rna_gt`)

In [ ]:
!pip install -q cooler h5py scipy matplotlib tqdm scikit-learn

import glob
npz_files = sorted(glob.glob('hicgan/data/rna_gt/*.npz'))
bw_files = sorted(glob.glob('hicgan/data/ctcf_H3K4me2_bigwig/*.bigwig'))
print(f'Found {len(npz_files)} RNA ground-truth .npz files and {len(bw_files)} .bigwig files: {bw_files}')
!cat hicgan/data/splits.txt

## 3. Clean Old TFRecords (Required When Switching Inputs)
Remove any old TFRecord files in `output_model` so new records are created using the `.bigwig` chromatin factor inputs (`CTCF` & `H3K4me2`).

In [ ]:
!rm -rf hicgan/output_model

## 4. Run Diffusion Model Training (`training.py`)

In [ ]:
!python hicgan/training.py \
    --dataDir hicgan/data \
    --splitFile hicgan/data/splits.txt \
    --windowSize 64 \
    --outputFolder hicgan/output_model \
    --epochs 50 \
    --batchSize 16 \
    --timesteps 1000 \
    --learningRatedenoiser 1e-4 \
    --plotFrequency 5 \
    --keepTFRecords

## 5. Predict & Visualize Test Chromosomes (`predict.py`)
Evaluate the trained model on the `test` chromosomes from `splits.txt` (or set `--split train` to evaluate on the exact same training chromosomes/random matrices).

In [ ]:
!python hicgan/predict.py \
    --trainedModel hicgan/output_model/denoiser_final.keras \
    --dataDir hicgan/data \
    --splitFile hicgan/data/splits.txt \
    --split test \
    --windowSize 64 \
    --batchSize 16 \
    --outputFolder hicgan/test_results

## 6. Display Training Loss and Visual Prediction Results

In [ ]:
import glob
from IPython.display import Image, display

loss_plot = 'hicgan/output_model/lossOverEpochs.png'
if os.path.exists(loss_plot):
    print('Loss Over Epochs:')
    display(Image(filename=loss_plot))

for fig_path in sorted(glob.glob('hicgan/test_results/pred_chr*.png')):
    print(fig_path)
    display(Image(filename=fig_path))